# 04b — Features de proporción (25 features)


In [1]:
import pandas as pd
import numpy as np
import re, time
from pathlib import Path

ROOT = Path.cwd().parents[1] if Path.cwd().name == 'fase4_gustos' else Path.cwd()
DATA_RAW = ROOT / 'data' / 'data_raw'
DATA_QC_GUSTOS = ROOT / 'data' / 'data_qc_gustos'
INFORMES = ROOT / 'informes' / 'fase4_gustos' / '02_features'
DATA_QC_GUSTOS.mkdir(parents=True, exist_ok=True)

REFERENCE_DATE = pd.Timestamp('2026-04-04', tz='UTC')

OID_RE = re.compile(r'[0-9a-f]{24}')
def clean_id(s):
    if pd.isna(s): return None
    m = OID_RE.search(str(s))
    return m.group(0) if m else None

sample = pd.read_parquet(DATA_QC_GUSTOS / 'sample_user_ids_gustos.parquet')
sample_ids = set(sample['user_id'])
N = len(sample)
features = pd.DataFrame({'user_id': sample['user_id'].values})
print(f'Sample N: {N:,}')

t0 = time.time()


Sample N: 114,412


In [2]:
# Mapping char→user (necesario para fights/arena)
chars = pd.read_csv(DATA_RAW/'characters.csv', usecols=['_id','user_id'], low_memory=False)
chars['char_id'] = chars['_id'].apply(clean_id)
chars['user_id'] = chars['user_id'].apply(clean_id)
chars = chars.dropna(subset=['char_id','user_id'])
char_to_user = dict(zip(chars['char_id'], chars['user_id']))
del chars
print(f'char_to_user: {len(char_to_user):,}')


char_to_user: 1,336,143


In [3]:
# === characters: pct por clase ===
print('[chars props]...')
t = time.time()
chars_df = pd.read_csv(DATA_RAW/'characters.csv', usecols=['user_id','class','arena_category','level'], low_memory=False)
chars_df['user_id'] = chars_df['user_id'].apply(clean_id)
chars_df = chars_df[chars_df['user_id'].isin(sample_ids)]

# pct por clase: 4 features (warrior, mage, archer, other)
class_counts = chars_df.groupby(['user_id','class']).size().unstack(fill_value=0)
class_total = class_counts.sum(axis=1)
for cls in ['warrior','mage','archer']:
    if cls in class_counts.columns:
        features[f'pct_class_{cls}'] = features['user_id'].map((class_counts[cls]/class_total).to_dict())
    else:
        features[f'pct_class_{cls}'] = 0.0
other_classes = [c for c in class_counts.columns if c not in ['warrior','mage','archer']]
features['pct_class_other'] = features['user_id'].map((class_counts[other_classes].sum(axis=1)/class_total).to_dict()) if other_classes else 0.0

# pct_chars_arena_active
arena_active = chars_df[chars_df['arena_category']>0].groupby('user_id').size()
total_chars = chars_df.groupby('user_id').size()
features['pct_chars_arena_active'] = features['user_id'].map((arena_active/total_chars).to_dict()).fillna(0)

# pct_chars_high_level (level >= p75 global)
p75 = chars_df['level'].quantile(0.75)
high = chars_df[chars_df['level']>=p75].groupby('user_id').size()
features['pct_chars_high_level'] = features['user_id'].map((high/total_chars).to_dict()).fillna(0)
del chars_df
print(f'[chars props] OK {time.time()-t:.1f}s')


[chars props]...


[chars props] OK 3.5s


In [4]:
# === items: pct critical/high_enhance/tempered ===
print('[items props]... chunks')
t = time.time()
n_total = {}
n_crit = {}
n_he = {}
n_temp = {}
for chunk in pd.read_csv(DATA_RAW/'user_items.csv',
                         usecols=['user_id','c_base_critical_chance','enhance_level','tempering_level'],
                         chunksize=2_000_000, low_memory=False):
    chunk['user_id'] = chunk['user_id'].apply(clean_id)
    chunk = chunk[chunk['user_id'].isin(sample_ids)]
    if len(chunk)==0: continue
    for u, n in chunk.groupby('user_id').size().items():
        n_total[u] = n_total.get(u,0) + n
    for u, n in chunk[chunk['c_base_critical_chance']>0].groupby('user_id').size().items():
        n_crit[u] = n_crit.get(u,0) + n
    for u, n in chunk[chunk['enhance_level']>=5].groupby('user_id').size().items():
        n_he[u] = n_he.get(u,0) + n
    for u, n in chunk[chunk['tempering_level']>0].groupby('user_id').size().items():
        n_temp[u] = n_temp.get(u,0) + n

features['pct_items_critical_build'] = features['user_id'].map({u: n_crit.get(u,0)/v for u,v in n_total.items()})
features['pct_items_high_enhance'] = features['user_id'].map({u: n_he.get(u,0)/v for u,v in n_total.items()})
features['pct_items_tempered'] = features['user_id'].map({u: n_temp.get(u,0)/v for u,v in n_total.items()})
print(f'[items props] OK {time.time()-t:.1f}s')


[items props]... chunks


[items props] OK 15.0s


In [5]:
# === currency: pct_inflow, pct_outflow, top_concept, pct_<currency> ===
print('[currency props]... 30d window, chunks')
t = time.time()
agg_total = {}
agg_in = {}
agg_out = {}
agg_currency = {}  # user -> currency -> count
agg_concept = {}   # user -> concept -> count
for chunk in pd.read_csv(DATA_RAW/'currency_transactions.csv',
                         usecols=['user_id','concept','currency','quantity'],
                         chunksize=1_000_000, low_memory=False):
    chunk['user_id'] = chunk['user_id'].apply(clean_id)
    chunk = chunk[chunk['user_id'].isin(sample_ids)]
    if len(chunk)==0: continue
    for u, g in chunk.groupby('user_id'):
        agg_total[u] = agg_total.get(u,0) + len(g)
        agg_in[u] = agg_in.get(u,0) + (g['quantity']>0).sum()
        agg_out[u] = agg_out.get(u,0) + (g['quantity']<0).sum()
        cur_counts = g['currency'].value_counts().to_dict()
        if u not in agg_currency:
            agg_currency[u] = {}
        for c, n in cur_counts.items():
            agg_currency[u][c] = agg_currency[u].get(c,0) + n
        conc_counts = g['concept'].value_counts().to_dict()
        if u not in agg_concept:
            agg_concept[u] = {}
        for c, n in conc_counts.items():
            agg_concept[u][c] = agg_concept[u].get(c,0) + n

features['currency_pct_inflow'] = features['user_id'].map({u: agg_in.get(u,0)/v for u,v in agg_total.items()})
features['currency_pct_outflow'] = features['user_id'].map({u: agg_out.get(u,0)/v for u,v in agg_total.items()})

# pct top1 concept
top_concept_pct = {}
for u, conc_d in agg_concept.items():
    if conc_d:
        top_concept_pct[u] = max(conc_d.values()) / sum(conc_d.values())
features['currency_pct_top1_concept'] = features['user_id'].map(top_concept_pct)

# pct por moneda
for cur in ['gold','gems','dark_steel']:
    features[f'currency_pct_{cur}'] = features['user_id'].map({u: d.get(cur,0)/agg_total.get(u,1) for u,d in agg_currency.items()})

# Guardar agg_concept/currency en globals para reuso en 04c (diversidad)
import pickle
with open(DATA_QC_GUSTOS / '_currency_agg_concept.pkl', 'wb') as f:
    pickle.dump(agg_concept, f)
with open(DATA_QC_GUSTOS / '_currency_agg_currency.pkl', 'wb') as f:
    pickle.dump(agg_currency, f)
with open(DATA_QC_GUSTOS / '_currency_agg_total.pkl', 'wb') as f:
    pickle.dump(agg_total, f)
print(f'[currency props] OK {time.time()-t:.1f}s, users con datos: {len(agg_total):,}')


[currency props]... 30d window, chunks


[currency props] OK 13.4s, users con datos: 21,351


In [6]:
# === fights: pct_pve, pct_pvp, pct_won, pct_validated ===
# fight_type values: FIGHT_TYPE_NODE (PvE campaign), FIGHT_TYPE_ARENA (PvP), FIGHT_TYPE_TOWER_EVENT
# fight_winner: 1.0 = player wins, 0.0 = enemy wins
# fight_validation_result: 1.0 = passed, -1.0 = failed
print('[fights props]... chunks')
t = time.time()
total = {}; pve = {}; pvp = {}; tower = {}; won = {}; validated = {}
type_counts = {}  # for entropy in 04c
for chunk in pd.read_csv(DATA_RAW/'fights_log.csv',
                         usecols=['player_id','fight_type','fight_winner','fight_validation_result'],
                         chunksize=500_000, low_memory=False):
    chunk['player_id'] = chunk['player_id'].apply(clean_id)
    chunk['user_id'] = chunk['player_id'].map(char_to_user)
    chunk = chunk.dropna(subset=['user_id'])
    chunk = chunk[chunk['user_id'].isin(sample_ids)]
    if len(chunk)==0: continue
    for u, g in chunk.groupby('user_id'):
        total[u] = total.get(u,0) + len(g)
        pve[u] = pve.get(u,0) + (g['fight_type']=='FIGHT_TYPE_NODE').sum()
        pvp[u] = pvp.get(u,0) + (g['fight_type']=='FIGHT_TYPE_ARENA').sum()
        tower[u] = tower.get(u,0) + (g['fight_type']=='FIGHT_TYPE_TOWER_EVENT').sum()
        won[u] = won.get(u,0) + (g['fight_winner']==1.0).sum()
        validated[u] = validated.get(u,0) + (g['fight_validation_result']==1.0).sum()
        if u not in type_counts:
            type_counts[u] = {}
        for ft, n in g['fight_type'].value_counts().items():
            type_counts[u][ft] = type_counts[u].get(ft,0) + n

features['fights_pct_pve'] = features['user_id'].map({u: pve.get(u,0)/v for u,v in total.items()})
features['fights_pct_pvp'] = features['user_id'].map({u: pvp.get(u,0)/v for u,v in total.items()})
features['fights_pct_won'] = features['user_id'].map({u: won.get(u,0)/v for u,v in total.items()})
features['fights_pct_validated'] = features['user_id'].map({u: validated.get(u,0)/v for u,v in total.items()})

# Persist fight type_counts for diversity
import pickle
with open(DATA_QC_GUSTOS/'_fights_type_counts.pkl','wb') as f:
    pickle.dump(type_counts, f)
with open(DATA_QC_GUSTOS/'_fights_total.pkl','wb') as f:
    pickle.dump(total, f)
print(f'[fights props] OK {time.time()-t:.1f}s, users con datos: {len(total):,}')


[fights props]... chunks


[fights props] OK 67.3s, users con datos: 20,588


In [7]:
# === collection recent, rewards premium track, devices pct_android ===
print('[other props]...')
t = time.time()

# collection 30d/60d ya están en 04g como coll_items_recent_30d/60d → re-calcular como %
coll = pd.read_csv(DATA_RAW/'user_items_collection.csv', usecols=['user_id','created_at'], low_memory=False)
coll['user_id'] = coll['user_id'].apply(clean_id)
coll = coll[coll['user_id'].isin(sample_ids)]
coll['ts'] = pd.to_datetime(coll['created_at'], errors='coerce', utc=True)
coll_total = coll.groupby('user_id').size()
coll_30d = coll[coll['ts']>=REFERENCE_DATE-pd.Timedelta(days=30)].groupby('user_id').size()
coll_60d = coll[coll['ts']>=REFERENCE_DATE-pd.Timedelta(days=60)].groupby('user_id').size()
features['coll_pct_recent_30d'] = features['user_id'].map((coll_30d/coll_total).to_dict()).fillna(0)
features['coll_pct_recent_60d'] = features['user_id'].map((coll_60d/coll_total).to_dict()).fillna(0)
del coll

# rewards premium track
rw = pd.read_csv(DATA_RAW/'user_daily_rewards.csv',
                 usecols=['user_id','last_claimed_premium_reward_day','last_claimed_ad_reward_day'], low_memory=False)
rw['user_id'] = rw['user_id'].apply(clean_id)
rw = rw[rw['user_id'].isin(sample_ids)].set_index('user_id')
rw['premium_pct'] = rw['last_claimed_premium_reward_day'].fillna(0) / (rw['last_claimed_premium_reward_day'].fillna(0) + rw['last_claimed_ad_reward_day'].fillna(0)).replace(0, np.nan)
features['reward_pct_premium_track'] = features['user_id'].map(rw['premium_pct'].to_dict())
del rw

# devices pct_android
dev = pd.read_csv(DATA_RAW/'devices.csv', usecols=['user_id','platform'], low_memory=False)
dev['user_id'] = dev['user_id'].apply(clean_id)
dev = dev[dev['user_id'].isin(sample_ids)]
total = dev.groupby('user_id').size()
android = dev[dev['platform'].astype(str).str.lower()=='android'].groupby('user_id').size()
features['device_pct_android'] = features['user_id'].map((android/total).to_dict()).fillna(0)
del dev
print(f'[other props] OK {time.time()-t:.1f}s')


[other props]...


[other props] OK 17.3s


In [8]:
# Save + report
assert len(features)==N and features['user_id'].is_unique
features.to_parquet(DATA_QC_GUSTOS / 'features_proporciones.parquet', index=False)
print(f'Guardado: {features.shape}')
print(f'Tiempo total: {time.time()-t0:.1f}s')
nan_rates = (features.isna().mean()*100).round(2)
print('\nNaN rates (top 15):')
print(nan_rates.sort_values(ascending=False).head(15).to_string())

rep_dir = INFORMES / '04b_proporciones'
rep_dir.mkdir(parents=True, exist_ok=True)
report = [f'# 04b - Proporciones\n', f'**Shape**: {features.shape}\n', f'**Tiempo**: {time.time()-t0:.1f}s\n', '\n| Feature | %NaN |', '|---|---:|']
for f, v in nan_rates.sort_values(ascending=False).items():
    report.append(f'| {f} | {v} |')
(rep_dir / 'execution_report.md').write_text('\n'.join(report))


Guardado: (114412, 24)
Tiempo total: 121.6s

NaN rates (top 15):
fights_pct_validated         82.01
fights_pct_won               82.01
fights_pct_pvp               82.01
fights_pct_pve               82.01
currency_pct_top1_concept    81.34
currency_pct_gold            81.34
currency_pct_dark_steel      81.34
currency_pct_inflow          81.34
currency_pct_outflow         81.34
currency_pct_gems            81.34
reward_pct_premium_track      4.91
coll_pct_recent_60d           0.00
coll_pct_recent_30d           0.00
user_id                       0.00
pct_class_warrior             0.00


820